# Chapter 7 — Working with Text Data

Chapter ini membahas representasi data teks menggunakan Bag-of-Words, TF-IDF, n-grams, dan topic modeling sederhana.

In [1]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Dataset Teks Sederhana

In [2]:
texts = [
 'film ini sangat bagus dan ceritanya menarik',
 'akting pemain sangat memukau dan alurnya bagus',
 'saya suka film ini karena visualnya indah',
 'cerita bagus musik bagus dan akhir yang memuaskan',
 'film ini buruk dan sangat membosankan',
 'alur cerita jelek akting buruk dan tidak menarik',
 'saya tidak suka film ini karena terlalu lambat',
 'visual buruk cerita membingungkan dan akhir mengecewakan'
]
labels = np.array([1,1,1,1,0,0,0,0])
for t,l in zip(texts, labels):
    print(l, '-', t)

1 - film ini sangat bagus dan ceritanya menarik
1 - akting pemain sangat memukau dan alurnya bagus
1 - saya suka film ini karena visualnya indah
1 - cerita bagus musik bagus dan akhir yang memuaskan
0 - film ini buruk dan sangat membosankan
0 - alur cerita jelek akting buruk dan tidak menarik
0 - saya tidak suka film ini karena terlalu lambat
0 - visual buruk cerita membingungkan dan akhir mengecewakan


## 2. Bag-of-Words dengan CountVectorizer

In [3]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
vect = CountVectorizer()
X_bow = vect.fit_transform(texts)
print('Shape matrix:', X_bow.shape)
print('Vocabulary:', vect.get_feature_names_out())
print('\nRepresentasi dense:')
print(pd.DataFrame(X_bow.toarray(), columns=vect.get_feature_names_out()))

Shape matrix: (8, 31)
Vocabulary: ['akhir' 'akting' 'alur' 'alurnya' 'bagus' 'buruk' 'cerita' 'ceritanya'
 'dan' 'film' 'indah' 'ini' 'jelek' 'karena' 'lambat' 'membingungkan'
 'membosankan' 'memuaskan' 'memukau' 'menarik' 'mengecewakan' 'musik'
 'pemain' 'sangat' 'saya' 'suka' 'terlalu' 'tidak' 'visual' 'visualnya'
 'yang']

Representasi dense:
   akhir  akting  alur  alurnya  bagus  buruk  cerita  ceritanya  dan  film  \
0      0       0     0        0      1      0       0          1    1     1   
1      0       1     0        1      1      0       0          0    1     0   
2      0       0     0        0      0      0       0          0    0     1   
3      1       0     0        0      2      0       1          0    1     0   
4      0       0     0        0      0      1       0          0    1     1   
5      0       1     1        0      0      1       1          0    1     0   
6      0       0     0        0      0      0       0          0    0     1   
7      1       0    

## 3. Klasifikasi Teks

In [4]:
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
model = make_pipeline(CountVectorizer(), LogisticRegression())
model.fit(texts, labels)
new_reviews = ['film bagus dan menarik', 'film buruk dan membosankan', 'akting bagus tetapi cerita lambat']
pred = model.predict(new_reviews)
for review, p in zip(new_reviews, pred):
    print(review, '->', 'positif' if p==1 else 'negatif')

film bagus dan menarik -> positif
film buruk dan membosankan -> negatif
akting bagus tetapi cerita lambat -> positif


## 4. TF-IDF dan n-Grams

In [5]:
tfidf_model = make_pipeline(TfidfVectorizer(), LogisticRegression())
tfidf_model.fit(texts, labels)
print('Prediksi TF-IDF:', tfidf_model.predict(new_reviews))
ngram = CountVectorizer(ngram_range=(1,2))
X_ngram = ngram.fit_transform(texts)
print('Jumlah fitur unigram + bigram:', len(ngram.get_feature_names_out()))
print('Contoh fitur:', ngram.get_feature_names_out()[:25])

Prediksi TF-IDF: [1 0 1]
Jumlah fitur unigram + bigram: 73
Contoh fitur: ['akhir' 'akhir mengecewakan' 'akhir yang' 'akting' 'akting buruk'
 'akting pemain' 'alur' 'alur cerita' 'alurnya' 'alurnya bagus' 'bagus'
 'bagus dan' 'bagus musik' 'buruk' 'buruk cerita' 'buruk dan' 'cerita'
 'cerita bagus' 'cerita jelek' 'cerita membingungkan' 'ceritanya'
 'ceritanya menarik' 'dan' 'dan akhir' 'dan alurnya']


## 5. Topic Modeling dengan NMF

In [6]:
from sklearn.decomposition import NMF
vec = TfidfVectorizer()
X_tfidf = vec.fit_transform(texts)
nmf = NMF(n_components=2, random_state=RANDOM_STATE, init='nndsvda', max_iter=500)
nmf.fit(X_tfidf)
features = vec.get_feature_names_out()
for i, topic in enumerate(nmf.components_):
    top = [features[j] for j in topic.argsort()[-5:][::-1]]
    print(f'Topik {i}:', ', '.join(top))

Topik 0: dan, bagus, sangat, buruk, cerita
Topik 1: saya, suka, karena, ini, film


## Kesimpulan

Data teks harus dikonversi menjadi fitur numerik. CountVectorizer, TF-IDF, n-grams, dan topic modeling adalah teknik dasar yang penting untuk NLP klasik.